# **I. Environment Setup on Google Colab**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **II. Model Definition and Feature Extraction**

In [ ]:
!git clone https://github.com/ilyas-sadiki/GBCosFace-.git
%cd GBCosFace-

fatal: destination path 'GBCosFace-' already exists and is not an empty directory.
/content/GBCosFace-


In [ ]:
!pip install -r requirements.txt
!pip install timm
!pip install optuna

  Using cached easydict-1.9.tar.gz (6.4 kB)
  Preparing metadata (setup.py) ... done
  Using cached mxnet-1.9.1-py3-none-manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from backbone.backbone_def import BackboneFactory
from head.head_def import HeadFactory
from loss.loss_def import LossFactory
import os
import yaml

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== Paths (adjust if needed) =====
SPLIT_DATASET_PATH = "/content/drive/MyDrive/PFE/datasets/dataset_split_job2"
TRAIN_PATH = os.path.join(SPLIT_DATASET_PATH, "train", "known")
VAL_PATH = os.path.join(SPLIT_DATASET_PATH, "val", "known")

PRETRAINED_MODEL_PATH = "/content/drive/MyDrive/PFE/models/gb_cosface_backbone.pth"
SAVE_MODEL_PATH = "/content/drive/MyDrive/PFE/models/finetuned_backbone_alpha025_sansEPS.pth"
SAVE_PROTOTYPES_PATH = "/content/drive/MyDrive/PFE/models/subcenter_prototypes_alpha025_sansEPS.pt"  # NEW

# ===== Dataset setup =====
NUM_CLASSES = len(os.listdir(TRAIN_PATH))

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=transform)
val_dataset = datasets.ImageFolder(VAL_PATH, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# ===== Load Backbone =====
def load_iresnet100(path):
    backbone_type = "iresnet100"
    backbone_conf_file = "/content/GBCosFace-/configs/backbone_conf.yaml"
    backbone_factory = BackboneFactory(backbone_type, backbone_conf_file)
    backbone = backbone_factory.get_backbone()
    state_dict = torch.load(path, map_location="cpu")
    backbone.load_state_dict(state_dict)
    for name, param in backbone.named_parameters():
        param.requires_grad = any(name.startswith(k) for k in ["layer4", "bn2", "dropout", "fc", "features"])
    return backbone

# ===== Validation =====
def validate(model, head, loss_fn, val_loader):
    model.eval()
    head.eval()
    total_loss = 0.0
    total_samples = 0
    correct = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            embeddings = model(imgs)
            logits = head(embeddings)
            loss = loss_fn(logits, labels)
            if isinstance(loss, dict):
                loss = sum(v for k, v in loss.items() if 'loss' in k)
            total_loss += loss.item() * imgs.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total_samples += imgs.size(0)

    avg_loss = total_loss / total_samples
    accuracy = correct / total_samples
    return avg_loss, accuracy

# ===== Training =====
def train(alpha=0.1, lr=0.01, optimizer_type='SGD'):
    model = load_iresnet100(PRETRAINED_MODEL_PATH).to(DEVICE)

    # --- Load and update head config ---
    head_conf_file = "/content/GBCosFace-/configs/head_conf.yaml"
    with open(head_conf_file) as f:
        head_conf = yaml.load(f, Loader=yaml.FullLoader)
    head_type = "SubCenterHead" if "SubCenterHead" in head_conf else "BaseHead"
    head_conf[head_type]["num_class"] = NUM_CLASSES
    with open(head_conf_file, "w") as f:
        yaml.dump(head_conf, f)

    head_factory = HeadFactory(head_type, head_conf_file)
    head = head_factory.get_head().to(DEVICE)

    # --- Load and patch loss config ---
    loss_conf_file = "/content/GBCosFace-/configs/loss_conf.yaml"
    with open(loss_conf_file) as f:
        loss_conf = yaml.load(f, Loader=yaml.FullLoader)
    loss_conf['GBCosFace']['alpha'] = alpha

    with open(loss_conf_file, "w") as f:
        yaml.dump(loss_conf, f)

    loss_type = "GBCosFace"
    loss_fn = LossFactory(loss_type, loss_conf_file).get_loss()

    params = list(filter(lambda p: p.requires_grad, model.parameters())) + list(head.parameters())
    if optimizer_type == 'SGD':
        optimizer = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    elif optimizer_type == 'Adam':
        optimizer = optim.Adam(params, lr=lr, weight_decay=5e-4)
    elif optimizer_type == 'AdamW':
        optimizer = optim.AdamW(params, lr=lr, weight_decay=5e-4)
    else:
        raise ValueError(f"Unknown optimizer type: {optimizer_type}")

    best_val_loss = float('inf')

    for epoch in range(10):
        model.train()
        head.train()
        running_loss = 0.0
        for i, (imgs, labels) in enumerate(train_loader):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            embeddings = model(imgs)
            logits = head(embeddings)

            loss_batch = loss_fn(logits, labels)
            loss_val = sum(v for k, v in loss_batch.items() if 'loss' in k) if isinstance(loss_batch, dict) else loss_batch

            optimizer.zero_grad()
            loss_val.backward()
            optimizer.step()
            running_loss += loss_val.item()

            if i % 10 == 0:
                print(f"[Epoch {epoch+1}/10] Step {i} Training Loss: {loss_val.item():.4f}")

        avg_train_loss = running_loss / len(train_loader)
        val_loss, val_acc = validate(model, head, loss_fn, val_loader)
        print(f"Epoch {epoch+1} Training Loss: {avg_train_loss:.4f} | Validation Loss: {val_loss:.4f} | Validation Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            os.makedirs(os.path.dirname(SAVE_MODEL_PATH), exist_ok=True)
            torch.save(model.state_dict(), SAVE_MODEL_PATH)
            print(f"✅ Saved best model at epoch {epoch+1}")

            if hasattr(head, "weight"):
                torch.save(head.weight.detach().cpu(), SAVE_PROTOTYPES_PATH)
                print(f"💾 Saved subcenter prototypes to: {SAVE_PROTOTYPES_PATH}")



# if __name__ == "__main__":
#     # === 1ST try ===
#     alpha = 0.25
#     eps = 0.03
#     lr = 0.001
#     optimizer_type = 'SGD'

#     print(f"\n🚀 Running with alpha={alpha}, eps={eps}, lr={lr}, optimizer={optimizer_type}")
#     train(alpha=alpha, eps=eps, lr=lr, optimizer_type=optimizer_type)


# if __name__ == "__main__":
#     # === 2ND try ===
#     alpha = 0.25
#     eps = 0.02
#     lr = 0.001
#     optimizer_type = 'SGD'

#     print(f"\n🚀 Running with alpha={alpha}, eps={eps}, lr={lr}, optimizer={optimizer_type}")
#     train(alpha=alpha, eps=eps, lr=lr, optimizer_type=optimizer_type)

if __name__ == "__main__":
    # === 3rd try ===
    alpha = 0.25
    lr = 0.001
    optimizer_type = 'SGD'

    print(f"\n🚀 Running with alpha={alpha}, lr={lr}, optimizer={optimizer_type}")
    train(alpha=alpha, lr=lr, optimizer_type=optimizer_type)


In [ ]:
import torch
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from backbone.backbone_def import BackboneFactory
from head.head_def import HeadFactory
import yaml
from sklearn.metrics.pairwise import cosine_similarity
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Paths ===
SPLIT_DATASET_PATH = "/content/drive/MyDrive/PFE/datasets/dataset_split_job2"
PROBE_KNOWN_PATH = os.path.join(SPLIT_DATASET_PATH, "test", "known")
PROBE_UNKNOWN_PATH = os.path.join(SPLIT_DATASET_PATH, "test", "unknown")

MODEL_PATH = "/content/drive/MyDrive/PFE/models/finetuned_backbone_alpha025_sansEPS.pth"
HEAD_CONF_FILE = "/content/GBCosFace-/configs/head_conf.yaml"
SUBCENTER_PROTOTYPES_PATH = "/content/drive/MyDrive/PFE/models/subcenter_prototypes_alpha025_sansEPS.pt"

BATCH_SIZE = 64

# === Data transforms (same as training) ===
transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def load_backbone(path):
    backbone_type = "iresnet100"
    backbone_conf_file = "/content/GBCosFace-/configs/backbone_conf.yaml"
    factory = BackboneFactory(backbone_type, backbone_conf_file)
    model = factory.get_backbone()
    state_dict = torch.load(path, map_location="cpu")
    model.load_state_dict(state_dict)
    model.to(DEVICE).eval()
    return model

def load_head():
    with open(HEAD_CONF_FILE) as f:
        head_conf = yaml.load(f, Loader=yaml.FullLoader)
    head_type = "SubCenterHead" if "SubCenterHead" in head_conf else "BaseHead"
    head_conf[head_type]["num_class"] = len(os.listdir(os.path.join(SPLIT_DATASET_PATH, "train", "known")))
    with open(HEAD_CONF_FILE, "w") as f:
        yaml.dump(head_conf, f)
    head_factory = HeadFactory(head_type, HEAD_CONF_FILE)
    head = head_factory.get_head()
    head.to(DEVICE).eval()
    return head

def extract_embeddings(dataloader, model):
    embeddings_list = []
    labels_list = []
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(DEVICE)
            embeddings = model(imgs)
            embeddings_list.append(embeddings.cpu().numpy())
            labels_list.append(labels.numpy())
    embeddings_all = np.vstack(embeddings_list)
    labels_all = np.concatenate(labels_list)
    return embeddings_all, labels_all

def extract_embeddings_unlabeled(dataloader, model):
    embeddings_list = []
    with torch.no_grad():
        for imgs, _ in dataloader:
            imgs = imgs.to(DEVICE)
            embeddings = model(imgs)
            embeddings_list.append(embeddings.cpu().numpy())
    embeddings_all = np.vstack(embeddings_list)
    return embeddings_all

# === Load datasets ===
probe_known_dataset = datasets.ImageFolder(PROBE_KNOWN_PATH, transform=transform)
probe_unknown_dataset = datasets.ImageFolder(PROBE_UNKNOWN_PATH, transform=transform)

probe_known_loader = DataLoader(probe_known_dataset, batch_size=BATCH_SIZE, shuffle=False)
probe_unknown_loader = DataLoader(probe_unknown_dataset, batch_size=BATCH_SIZE, shuffle=False)

# === Load model & head ===
print("Loading model and head...")
model = load_backbone(MODEL_PATH)
head = load_head()

# === Load subcenter prototypes (gallery embeddings) ===
print("Loading subcenter prototypes as gallery embeddings...")
subcenter_prototypes = torch.load(SUBCENTER_PROTOTYPES_PATH, map_location="cpu")  # Shape: (num_prototypes, embedding_dim)

# Normalize prototypes to unit length (important for cosine similarity)
gallery_emb = subcenter_prototypes.numpy()
gallery_emb /= np.linalg.norm(gallery_emb, axis=1, keepdims=True)

print(f"Number of prototypes in gallery: {gallery_emb.shape[0]}")

# === Extract probe embeddings ===
print("Extracting probe known embeddings...")
probe_known_emb, probe_known_ids = extract_embeddings(probe_known_loader, model)

print("Extracting probe unknown embeddings...")
probe_unknown_emb = extract_embeddings_unlabeled(probe_unknown_loader, model)

# Normalize probe embeddings to unit length
probe_known_emb /= np.linalg.norm(probe_known_emb, axis=1, keepdims=True)
probe_unknown_emb /= np.linalg.norm(probe_unknown_emb, axis=1, keepdims=True)

# === Match probe embeddings against gallery prototypes ===
print("Computing similarity scores...")
known_sim = cosine_similarity(probe_known_emb, gallery_emb)
unknown_sim = cosine_similarity(probe_unknown_emb, gallery_emb)

# For each probe, get highest similarity and index of prototype
top_scores_known = np.max(known_sim, axis=1)
top_prototype_indices_known = np.argmax(known_sim, axis=1)

top_scores_unknown = np.max(unknown_sim, axis=1)

# === Map prototype index to class id ===
# Assuming prototypes are arranged in order by class (subcenter heads typically do this)
# For example, if each class has M subcenters, then
# class_id = prototype_index // M

# You need to know subcenter count per class from head config
num_classes = len(os.listdir(os.path.join(SPLIT_DATASET_PATH, "train", "known")))

with open(HEAD_CONF_FILE) as f:
    head_conf = yaml.load(f, Loader=yaml.FullLoader)
head_type = "SubCenterHead" if "SubCenterHead" in head_conf else "BaseHead"
if head_type == "SubCenterHead":
    M = head_conf[head_type].get("num_subcenters", 3)  # Adjust if different in your config
else:
    M = 1

predicted_classes = top_prototype_indices_known // M

# === Rank-1 Accuracy ===
rank1_correct = np.sum(probe_known_ids == predicted_classes)
rank1_total = len(probe_known_ids)
rank1_accuracy = rank1_correct / rank1_total
print(f"\n🎯 Rank-1 Accuracy (Known Probes): {rank1_accuracy:.4f}")

# === TPIR @ FPIR ===
def compute_tpir_fpirs(tp_scores, fp_scores, fpir_targets):
    thresholds = np.sort(fp_scores)[::-1]  # Descending
    n_fp_total = len(fp_scores)
    n_tp_total = len(tp_scores)

    results = {}
    for fpir_target in fpir_targets:
        max_fp = int(np.floor(fpir_target * n_fp_total))
        if max_fp == 0:
            max_fp = 1

        thresh_idx = max_fp - 1
        if thresh_idx >= len(thresholds):
            thresh_idx = len(thresholds) - 1

        threshold = thresholds[thresh_idx]

        tp = np.sum(tp_scores >= threshold)
        tpir = tp / n_tp_total
        results[fpir_target] = {
            "threshold": float(threshold),
            "TPIR": float(tpir)
        }
    return results

fpir_targets = [1e-1, 1e-2, 1e-3, 1e-4]
results = compute_tpir_fpirs(top_scores_known, top_scores_unknown, fpir_targets)

print("\n📊 TPIR @ FPIR Results:")
for fpir, res in results.items():
    print(f"  TPIR @ FPIR={fpir:.0e}: {res['TPIR']:.4f} (Threshold: {res['threshold']:.4f})")
